# Store Sales · All available dates

This notebook uses all available training dates through each cutoff. Comparing its fixed-model score with the recent-year score shows how the training window affects the forecast.

Set **PREDICTOR_CHOICE** in the first code cell, then run the notebook from top to bottom.

- **Fixed predictors** fits Bella's model with promotions, holidays, events and closed days.
- **Stepwise selection** uses F1–F3 to choose from the calendar and promotion predictors below.

Place the original competition CSVs in `kaggle_comp/data/`. Both notebooks can also read notebook 02's `processed.parquet`. Keep `holidays_events.csv` there for the detailed event types. An older processed file also needs `stores.csv` so the notebook can rebuild the calendar. To install the notebook dependencies in your chosen environment, run `python -m pip install -e ".[dev,kaggle]"` from the repository root.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code/backtest/kaggle.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared_code.backtest import kaggle as bt
from kaggle_comp.src.data.load_data import load_sales_panel
from kaggle_comp.src.kaggle_pred import MODES, choose_predictors, forecast_period, prepare_run
from kaggle_comp.src.results import compare_reference, compare_windows, make_submission, plot_example, save_validation

PREDICTOR_CHOICE = "Fixed predictors"  # Or "Stepwise selection".
TRAIN_DAYS = None
WORKERS = min(4, os.cpu_count() or 1)
MAKE_SUBMISSION = True  # Set False to stop after validation and the window comparison.

# These three settings apply to stepwise selection.
SERIES_PER_GROUP = 2  # Two series from each of four sales groups. None uses every series.
MAX_STEPS = 4
MIN_IMPROVEMENT = 0.0005  # Required drop in selection RMSLE at each step.

DATA = ROOT / "kaggle_comp/data"
RESULTS = ROOT / "kaggle_comp/results/bella_settings"
OUT = prepare_run(RESULTS, TRAIN_DAYS, PREDICTOR_CHOICE)
print("Results folder:", OUT.relative_to(ROOT))

## 1. Read the sales and predictors

Each row is one store, product family and day. `closed` marks January 1 and December 25. `holiday` uses the observed holiday date and the store's location. `event` marks dates listed as events. Promotions and calendar dates are available for the 16 forecast days.

In [ ]:
panel = load_sales_panel(DATA)
print(f"{len(panel):,} rows; {panel.groupby(bt.K, observed=True).ngroups:,} store–family series")
display(panel[["store_nbr", "family", "date", "sales", "onpromotion", "holiday", "event", "closed"]].head())

## 2. Choose the predictors

**Fixed predictors.** Bella fits this regression separately for each store and product family:

$$
\log(1+\mathrm{sales}_t)=\beta_0 + \beta_1 \log(1+\mathrm{onpromotion}_t)
+ \beta_2\,\mathrm{holiday}_t + \beta_3\,\mathrm{event}_t + \beta_4\,\mathrm{closed}_t + u_t.
$$

AutoARIMA chooses the orders for the errors $u_t$ by BIC, with a weekly period of seven days. It includes a mean or drift when the selected model calls for one. A constant predictor is dropped for that series. The fixed fit includes the daily grid from notebook 02, including the zero-filled Christmas dates. A series with no sales in the last 30 days gets a zero forecast.

**Stepwise selection.** Start with six weekday indicators (Monday is the reference). Score the allowed changes on the same series and three 16-day periods. Accept a change when it lowers pooled RMSLE by at least `MIN_IMPROVEMENT`. A later step can drop a group. `MAX_STEPS` limits the number of accepted changes. This is a shorter search than trying every combination and can miss a combination that works well together.

| Candidate | Meaning |
|---|---|
| Promotion and previous day's promotion | A promotion may affect sales today and tomorrow. |
| Payday and day after payday | The 15th and month end mark payday. The 16th and the 1st mark the following day. A weekend term tests whether that adjustment differs on Saturday or Sunday. |
| Holiday and nearby days | Use the observed holiday, the day before or after, or the national, regional and local holiday flags. |
| Closed day and make-up workday | Allow the two special dates or an officially declared working day to have different sales. |
| Events and earthquake | Try the general event flag or more specific event types. The earthquake flag covers April 16 through May 15, 2016. It drops out when the training window starts later. |
| Trend and annual seasonality | Try a linear trend and up to two annual sine/cosine pairs. |

Both choices use `AutoARIMA(season_length=7, ic="bic")`, matching notebook 04. AutoARIMA chooses the differencing and ARIMA orders separately for each series with its default search limits. The six weekday indicators stay in the candidate predictor lists. Their coefficients need care when the fitted model uses weekly differencing, because weekday indicators repeat every seven days.

The selection fit uses sales after each store starts selling and fills missing Christmas observations from earlier weekdays. A short series, an exception during fitting, or an invalid forecast uses the four-week weekday baseline. The fit notes in `validation_fits.csv` record fitting messages. The daily forecast errors on F1–F3 determine the predictor choice.

In [ ]:
label, columns, selection_score = choose_predictors(
    panel, TRAIN_DAYS, PREDICTOR_CHOICE, OUT, series_per_group=SERIES_PER_GROUP,
    workers=WORKERS, max_steps=MAX_STEPS, min_improvement=MIN_IMPROVEMENT,
)
print("Predictors:", ", ".join(columns))
if selection_score is not None:
    display(pd.read_csv(OUT / "selection_path.csv")[["selection_step", "candidate", "selection_rmsle"]])

## 3. Forecast the final holdout

Fit through July 30, 2017 and forecast July 31 through August 15. The final validation RMSLE pools all 28,512 sales predictions. Lower is better. The F1–F3 score answers a different question about three earlier periods.

In [ ]:
validation, fits = forecast_period(
    panel, bt.HOLDOUT["HOLDOUT"], TRAIN_DAYS, PREDICTOR_CHOICE, columns, workers=WORKERS,
)
score = save_validation(validation, OUT, label, PREDICTOR_CHOICE, TRAIN_DAYS, selection_score)
if fits is not None:
    fits.to_csv(OUT / "validation_fits.csv", index=False)
display(pd.DataFrame([score]).round(5))

### Read the errors

The daily table shows RMSLE across stores and families on each forecast day. The example plot follows store 44's BEVERAGES series on the log scale used by RMSLE.

In [ ]:
display(pd.read_csv(OUT / "validation_by_day.csv").round(4))
plot_example(validation, OUT);

## 4. Compare the training windows

After both notebooks have scored the same predictor choice, this table shows their holdout scores. The family table shows the difference as all-history RMSLE minus recent-year RMSLE. A positive value favors the recent year.

In [ ]:
window_scores, family_scores = compare_windows(RESULTS, MODES[PREDICTOR_CHOICE])
if window_scores.empty:
    print("Run the other notebook with the same predictor choice to see both scores.")
else:
    display(window_scores.round(5))
    display(family_scores.round(4))

## 5. Make the Kaggle submission

Fit through August 15, 2017 and forecast August 16–31 using the chosen predictors. The file `submission.csv` contains one forecast for each Kaggle test ID.

In [ ]:
if MAKE_SUBMISSION:
    test_forecasts, test_fits = forecast_period(
        panel, bt.TEST_CUTOFF, TRAIN_DAYS, PREDICTOR_CHOICE, columns, workers=WORKERS,
    )
    if test_fits is not None:
        test_fits.to_csv(OUT / "test_fits.csv", index=False)
    test_forecasts.to_parquet(OUT / "test_predictions.parquet", index=False)
    submission = make_submission(test_forecasts, DATA, OUT)
    print(f"Saved {len(submission):,} rows to {OUT / 'submission.csv'}")
    display(submission.head())